# Notebook 06 — Comparative Evaluation
Loads all trained models (LSTM, GRU, CNN, Transformer, RF), produces the final comparison table, ROC overlay, and all figures.

In [ ]:
import subprocess, os
from pathlib import Path

REPO_URL  = 'https://github.com/calvinkatoroy/tugas-akhir-ai-kel-08.git'
REPO_NAME = 'tugas-akhir-ai-kel-08'

cwd = Path.cwd()

if (cwd / '../src').resolve().exists():
    result = subprocess.run(['git', '-C', str((cwd / '..').resolve()), 'pull'],
                            capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')

elif (cwd / 'src').exists():
    result = subprocess.run(['git', 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or 'Already up to date.')
    os.chdir('notebooks')

else:
    repo_path = cwd / REPO_NAME
    if not repo_path.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    else:
        print('Repo found, pulling latest...')
        subprocess.run(['git', '-C', str(repo_path), 'pull'], capture_output=True)
    os.chdir(repo_path / 'notebooks')

print(f'Working dir: {Path.cwd()}')

In [ ]:
import sys
sys.path.insert(0, "..")

import random
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

figures_dir = Path("../results/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/tugas-akhir-ai")
    SPLITS_DIR = DRIVE_ROOT / "splits"
else:
    import yaml
    with open("../config.yaml") as f:
        _cfg = yaml.safe_load(f)
    DRIVE_ROOT = Path(_cfg["data"]["drive_root"])
    SPLITS_DIR = Path(_cfg["data"]["splits_drive"])

print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"SPLITS_DIR: {SPLITS_DIR}")

In [ ]:
# LSTM/GRU/CNN/Transformer use seq splits; RF uses flat splits
X_test_seq  = np.load(SPLITS_DIR / "X_test_seq.npy")
y_test_seq  = np.load(SPLITS_DIR / "y_test_seq.npy")
X_test_flat = np.load(SPLITS_DIR / "X_test.npy")
y_test_flat = np.load(SPLITS_DIR / "y_test.npy")

n_features = X_test_seq.shape[2]
print("Seq test:", X_test_seq.shape, "  Flat test:", X_test_flat.shape)

## Load Models

In [ ]:
import shutil
from pathlib import Path

# Restore checkpoints from Drive to local results/
ckpt_dir = Path("../results/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)
for fname in ["best_lstm.pt", "best_gru.pt", "best_cnn.pt", "best_transformer.pt", "best_rf.pkl"]:
    src_path = DRIVE_ROOT / fname
    dst_path = ckpt_dir / fname
    if src_path.exists():
        if not dst_path.exists():
            shutil.copy(str(src_path), str(dst_path))
            print(f"Restored {fname} from Drive")
        else:
            print(f"{fname} already exists locally")
    else:
        print(f"WARNING: {fname} not found in Drive — skipping")

from src.models.lstm_model        import build_lstm
from src.models.gru_model         import build_gru
from src.models.cnn_model         import build_cnn
from src.models.transformer_model import build_transformer
from src.models.rf_model          import load_rf

lstm = build_lstm(cfg, n_features).to(device)
lstm.load_state_dict(torch.load(str(ckpt_dir / "best_lstm.pt"), map_location=device))
lstm.eval()

gru = build_gru(cfg, n_features).to(device)
gru.load_state_dict(torch.load(str(ckpt_dir / "best_gru.pt"), map_location=device))
gru.eval()

cnn = build_cnn(cfg, n_features).to(device)
cnn.load_state_dict(torch.load(str(ckpt_dir / "best_cnn.pt"), map_location=device))
cnn.eval()

transformer = build_transformer(cfg, n_features).to(device)
transformer.load_state_dict(torch.load(str(ckpt_dir / "best_transformer.pt"), map_location=device))
transformer.eval()

rf = load_rf(str(ckpt_dir / "best_rf.pkl"))

print("All models loaded.")

## Per-Model Evaluation

In [ ]:
from src.evaluate import evaluate_dl_model, evaluate_rf_model
from sklearn.metrics import roc_curve, auc

_, y_prob_lstm = evaluate_dl_model(
    lstm, X_test_seq, y_test_seq, model_name='LSTM',
    figures_dir='../results/figures', csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

_, y_prob_gru = evaluate_dl_model(
    gru, X_test_seq, y_test_seq, model_name='GRU',
    figures_dir='../results/figures', csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

_, y_prob_cnn = evaluate_dl_model(
    cnn, X_test_seq, y_test_seq, model_name='CNN',
    figures_dir='../results/figures', csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

_, y_prob_tr = evaluate_dl_model(
    transformer, X_test_seq, y_test_seq, model_name='Transformer',
    figures_dir='../results/figures', csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

_, y_prob_rf = evaluate_rf_model(
    rf, X_test_flat, y_test_flat,
    figures_dir='../results/figures', csv_path=str(DRIVE_ROOT / "metrics_summary.csv"))

## ROC Curve Overlay

In [ ]:
from src.utils import plot_roc_overlay

fpr_lstm, tpr_lstm, _ = roc_curve(y_test_seq,  y_prob_lstm, pos_label=1)
fpr_gru,  tpr_gru,  _ = roc_curve(y_test_seq,  y_prob_gru,  pos_label=1)
fpr_cnn,  tpr_cnn,  _ = roc_curve(y_test_seq,  y_prob_cnn,  pos_label=1)
fpr_tr,   tpr_tr,   _ = roc_curve(y_test_seq,  y_prob_tr,   pos_label=1)
fpr_rf,   tpr_rf,   _ = roc_curve(y_test_flat, y_prob_rf,   pos_label=1)

roc_data = {
    'LSTM':        (fpr_lstm, tpr_lstm, auc(fpr_lstm, tpr_lstm)),
    'GRU':         (fpr_gru,  tpr_gru,  auc(fpr_gru,  tpr_gru)),
    'CNN':         (fpr_cnn,  tpr_cnn,  auc(fpr_cnn,  tpr_cnn)),
    'Transformer': (fpr_tr,   tpr_tr,   auc(fpr_tr,   tpr_tr)),
    'RF':          (fpr_rf,   tpr_rf,   auc(fpr_rf,   tpr_rf)),
}

plot_roc_overlay(roc_data, save_path='../results/figures/roc_comparison.png')

## Final Comparison Table

In [ ]:
summary = pd.read_csv(str(DRIVE_ROOT / "metrics_summary.csv"))

# Keep only final test-set evaluation rows (have roc_auc filled)
test_rows = summary[summary['model'].isin(['LSTM', 'GRU', 'CNN', 'Transformer', 'RandomForest'])].copy()
test_rows = test_rows.dropna(subset=['roc_auc'])

display_cols = ['model', 'accuracy', 'precision_ddos', 'recall_ddos', 'f1_ddos', 'fpr', 'roc_auc']
final_table = test_rows[display_cols].drop_duplicates(subset='model').set_index('model')
print(final_table.to_string())
final_table.to_csv('../results/final_comparison.csv')
print('\nSaved to results/final_comparison.csv')

## Hyperparameter Tuning Summary

In [ ]:
dl_cols = [c for c in ['exp_id', 'hidden_size', 'd_model', 'num_filters', 'num_layers', 'dropout', 'best_val_f1', 'notes'] if c in summary.columns]
rf_cols  = [c for c in ['exp_id', 'best_val_f1', 'notes'] if c in summary.columns]

for model_name in ['LSTM', 'GRU', 'CNN', 'Transformer']:
    print(f"=== {model_name} experiments ===")
    subset = summary[summary['model'] == model_name]
    avail = [c for c in dl_cols if c in subset.columns]
    print(subset[avail].to_string(index=False))
    print()

print("=== RF experiments ===")
print(summary[summary['model'] == 'RandomForest'][rf_cols].to_string(index=False))

## Bar Chart — F1 / AUC Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

models = final_table.index.tolist()
colors = ['#4C72B0', '#55A868', '#DD8452', '#8172B2', '#C44E52']

axes[0].bar(models, final_table['f1_ddos'], color=colors[:len(models)])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('F1 Score (DDoS class)')
axes[0].set_title('F1 Score Comparison')
for i, v in enumerate(final_table['f1_ddos']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=8)

axes[1].bar(models, final_table['roc_auc'], color=colors[:len(models)])
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('ROC-AUC')
axes[1].set_title('ROC-AUC Comparison')
for i, v in enumerate(final_table['roc_auc']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison — CIC-DDoS2019', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/model_comparison_bar.png', dpi=150)
plt.show()